# Order Funnel Analysis

Analyses order volume, fulfillment rates, and status distribution across sources.

**Data source:** `marts.fct_orders`  
**Last updated:** 2026-03-06

---

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(dotenv_path='../../.env')
engine = create_engine(os.environ['DATABASE_URL'])

## 1. Orders by Source (MTD)

In [ ]:
orders_by_source = pd.read_sql(
    """
    select
        source,
        count(*)                                   as total_orders,
        count(*) filter (where order_status = 'fulfilled') as fulfilled_orders,
        round(
            100.0 * count(*) filter (where order_status = 'fulfilled')
                  / nullif(count(*), 0),
            1
        )                                          as fulfillment_rate_pct,
        round(avg(order_total_usd), 2)             as avg_order_value_usd
    from marts.fct_orders
    where order_date >= date_trunc('month', current_date)
    group by 1
    order by total_orders desc
    """,
    engine,
)

orders_by_source

## 2. Weekly Order Volume Trend

In [ ]:
weekly_orders = pd.read_sql(
    """
    select
        order_week,
        source,
        count(*) as orders
    from marts.fct_orders
    where order_week >= current_date - interval '16 weeks'
    group by 1, 2
    order by 1, 2
    """,
    engine,
    parse_dates=['order_week'],
)

pivot = weekly_orders.pivot_table(
    index='order_week', columns='source', values='orders', aggfunc='sum'
).fillna(0)

pivot.plot(kind='area', stacked=True, figsize=(14, 5), colormap='tab10', alpha=0.75)
plt.title('Weekly Order Volume by Source (Last 16 Weeks)')
plt.xlabel('')
plt.ylabel('Orders')
plt.tight_layout()
plt.show()